<a href="https://colab.research.google.com/github/Dariap5/music-finetuning/blob/main/musif_finttuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# 0. ОЧИСТКА И УСТАНОВКА
# ==========================================
!fuser -k 8000/tcp 2>/dev/null
!pip install -q git+https://github.com/suno-ai/bark.git
!pip install -q transformers==4.44.2 accelerate scipy fastapi uvicorn pyngrok pydantic

import os
import re
import gc
import threading
import traceback
import contextlib
import numpy as np
import torch
import scipy.io.wavfile as wavfile
import scipy.signal as signal
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoProcessor,
    MusicgenMelodyForConditionalGeneration,
)
from bark import SAMPLE_RATE as BARK_SR, generate_audio, preload_models
from bark.generation import clean_models as bark_clean_models
from fastapi import FastAPI, HTTPException
from fastapi.responses import FileResponse
from pydantic import BaseModel, Field
import uvicorn
from pyngrok import ngrok
from google.colab import userdata

# ==========================================
# 1. ПАТЧ БЕЗОПАСНОСТИ ВЕСОВ
# ==========================================
if not hasattr(torch, "_original_load"):
    torch._original_load = torch.load
    def _patched_load(*args, **kwargs):
        kwargs.setdefault("weights_only", False)
        return torch._original_load(*args, **kwargs)
    torch.load = _patched_load

# ==========================================
# 2. КОНФИГУРАЦИЯ
# ==========================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
MUSICGEN_SR = 32000
OUTPUT_SR = 32000
OUTPUT_FILE = "final_rock_hit.wav"
MAX_DURATION_SEC = 15
TARGET_DURATION_SEC = 12

QWEN_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"   # обратно на 1.5B
MUSICGEN_MODEL = "facebook/musicgen-melody"

print(f"💻 Device: {DEVICE} | dtype: {DTYPE}")

# ==========================================
# 3. УТИЛИТЫ ПАМЯТИ
# ==========================================
def free_vram():
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    gc.collect()


def log_vram(stage: str):
    if DEVICE == "cuda":
        used = torch.cuda.memory_allocated() / 1e9
        peak = torch.cuda.max_memory_allocated() / 1e9
        print(f"  💾 [{stage}] VRAM used={used:.2f} GB | peak={peak:.2f} GB")


@contextlib.contextmanager
def on_gpu(model):
    """Перемещает модель на GPU на время блока, потом возвращает на CPU."""
    try:
        model.to(DEVICE)
        free_vram()
        yield model
    finally:
        model.to("cpu")
        free_vram()

# ==========================================
# 4. ЗАГРУЗКА МОДЕЛЕЙ (ВСЕ НА CPU)
# ==========================================
print("🚀 Загружаю модели (3-5 минут)...")

# --- Qwen на CPU ---
qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL)
qwen_model = AutoModelForCausalLM.from_pretrained(
    QWEN_MODEL,
    torch_dtype=DTYPE,
).to("cpu")
qwen_model.eval()

# --- Bark: модели грузятся внутрь bark.generation, управляются им сам ---
preload_models()

# --- MusicGen-Melody на CPU (fp32 — иначе RuntimeError на melody-conditioning) ---
music_processor = AutoProcessor.from_pretrained(MUSICGEN_MODEL)
music_model = MusicgenMelodyForConditionalGeneration.from_pretrained(
    MUSICGEN_MODEL,
    torch_dtype=torch.float32,   # ← ВАЖНО: было DTYPE (=fp16)
).to("cpu")
music_model.eval()
if hasattr(music_model, "enable_attention_slicing"):
    music_model.enable_attention_slicing()
free_vram()
log_vram("after load")
print("✅ Студия готова!\n")

# ==========================================
# 5. AUDIO УТИЛИТЫ
# ==========================================
def resample_audio(audio: np.ndarray, src_sr: int, dst_sr: int) -> np.ndarray:
    if src_sr == dst_sr:
        return audio.astype(np.float32)
    new_len = int(round(len(audio) * dst_sr / src_sr))
    return signal.resample(audio, new_len).astype(np.float32)


def normalize(audio: np.ndarray, peak: float = 0.95) -> np.ndarray:
    max_val = np.max(np.abs(audio))
    if max_val < 1e-8:
        return audio
    return (audio / max_val * peak).astype(np.float32)

# ==========================================
# 6. ШАГИ ПАЙПЛАЙНА
# ==========================================
def generate_lyrics(topic: str) -> str:
    """Qwen → 4 рифмованные строки."""
    print("📝 Пишу текст...")
    clean = re.sub(r"[\n\r\t]+", " ", topic).strip()[:200]
    messages = [
        {
            "role": "system",
            "content": (
                "Ты — рок-поэт. Пиши ТОЛЬКО текст песни на русском языке. "
                "Ровно 4 короткие строки. Рифма обязательна (схема ABAB или AABB). "
                "Каждая строка — 5-8 слов, энергичная, образная. "
                "НЕ пиши пояснений, заголовков, нумерации — только сам текст песни."
            ),
        },
        {
            "role": "user",
            "content": (
                f"Тема: {clean}\n\n"
                "Напиши 4 рифмованные рок-строки. Только текст, ничего лишнего."
            ),
        },
    ]
    prompt = qwen_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    with on_gpu(qwen_model) as model:
        inputs = qwen_tokenizer(prompt, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=150,
                do_sample=True,
                temperature=0.85,
                top_p=0.92,
                repetition_penalty=1.15,   # против повторов
                pad_token_id=qwen_tokenizer.eos_token_id,
            )
        text = qwen_tokenizer.decode(
            out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
        ).strip()
        del inputs, out

    # Чистим от кавычек, нумерации, разметки
    text = re.sub(r"^[\*\-#\d\.\)\s\"«»]+", "", text, flags=re.MULTILINE)
    text = re.sub(r"[\"«»]+", "", text)
    lines = [ln.strip() for ln in text.split("\n") if ln.strip()]
    # Берём первые 4 непустых строки
    lines = lines[:4]
    result = " / ".join(lines) if lines else clean
    print(f"   → {result}")
    return result

def generate_vocals(lyrics: str) -> np.ndarray:
    """Bark → вокал @ 32 kHz. Bark сам управляет своими моделями на GPU."""
    print("🎤 Записываю вокал...")
    audio = generate_audio(f"♪ {lyrics} ♪", history_prompt="v2/ru_speaker_6")
    audio = audio.astype(np.float32)

    # После Bark освобождаем его VRAM
    try:
        bark_clean_models()
    except Exception:
        pass
    free_vram()

    return resample_audio(audio, BARK_SR, MUSICGEN_SR)


def generate_music(vocal_32k: np.ndarray, style_prompt: str,
                   target_duration_sec: float = 12.0) -> np.ndarray:
    """MusicGen-Melody → аранжировка @ 32 kHz."""
    print(f"🎸 Генерирую аранжировку (~{target_duration_sec:.0f} сек)...")

    target_samples = int(target_duration_sec * MUSICGEN_SR)

    # Если вокал короче цели — паддим тишиной справа
    if len(vocal_32k) < target_samples:
        pad = np.zeros(target_samples - len(vocal_32k), dtype=np.float32)
        vocal_32k = np.concatenate([vocal_32k, pad])
    else:
        vocal_32k = vocal_32k[:target_samples]

    duration_sec = len(vocal_32k) / MUSICGEN_SR
    max_new_tokens = min(int(duration_sec * 50) + 50, 1000)

    inputs = music_processor(
        text=[style_prompt],
        audio=vocal_32k,
        sampling_rate=MUSICGEN_SR,
        return_tensors="pt",
        padding=True,
    )

    with on_gpu(music_model) as model:
        target_dtype = next(model.parameters()).dtype
        gpu_inputs = {}
        for k, v in inputs.items():
            if torch.is_tensor(v):
                if v.dtype.is_floating_point:
                    gpu_inputs[k] = v.to(device=DEVICE, dtype=target_dtype)
                else:
                    gpu_inputs[k] = v.to(DEVICE)
            else:
                gpu_inputs[k] = v

        with torch.no_grad():
            tokens = model.generate(
                **gpu_inputs,
                max_new_tokens=max_new_tokens,
                guidance_scale=3.0,
                do_sample=True,
                temperature=1.0,
            )
        music_audio = tokens[0, 0].cpu().float().numpy()
        del tokens, gpu_inputs, inputs

    return music_audio


def mix_tracks(vocal: np.ndarray, music: np.ndarray,
               vocal_gain: float = 1.0, music_gain: float = 0.55) -> np.ndarray:
    print("🎚️ Свожу...")
    length = min(len(vocal), len(music))
    mix = vocal[:length] * vocal_gain + music[:length] * music_gain
    return normalize(mix)

# ==========================================
# 7. ГЛАВНЫЙ PIPELINE
# ==========================================
ROCK_STYLE = (
    "Modern Russian rock, heavy distorted electric guitars, "
    "powerful drums, driving bass, stadium energy, anthemic chorus"
)


def create_rock_track(topic: str, duration_sec: float = TARGET_DURATION_SEC) -> dict:
    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()

    log_vram("start")

    lyrics = generate_lyrics(topic)
    log_vram("after lyrics")

    vocal = generate_vocals(lyrics)
    log_vram("after vocals")

    music = generate_music(vocal, ROCK_STYLE, target_duration_sec=duration_sec)
    log_vram("after music")

    # Паддим вокал до длины музыки тишиной (если он короче)
    if len(vocal) < len(music):
        pad = np.zeros(len(music) - len(vocal), dtype=np.float32)
        vocal = np.concatenate([vocal, pad])

    final = mix_tracks(vocal, music)
    wavfile.write(OUTPUT_FILE, OUTPUT_SR, final)

    duration = len(final) / OUTPUT_SR
    print(f"✅ Готово: {duration:.1f} сек\n")
    return {"lyrics": lyrics, "duration_sec": round(duration, 2)}


# ==========================================
# 8. FASTAPI
# ==========================================
# ==========================================
# 8. FASTAPI
# ==========================================
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("rockgen")

app = FastAPI(title="Rock Generator", version="3.1")


class GenRequest(BaseModel):
    topic: str = Field(..., min_length=1, max_length=200, examples=["Большие города"])
    duration_sec: float = Field(12.0, ge=4.0, le=15.0, examples=[12.0])


_gen_lock = threading.Lock()


@app.post("/generate")
def api_generate(data: GenRequest):
    logger.info(f"📥 Request: topic={data.topic!r} duration={data.duration_sec}")

    if not _gen_lock.acquire(blocking=False):
        logger.warning("⚠️  Lock busy — rejecting")
        raise HTTPException(status_code=429, detail="Already generating, try again later")

    try:
        result = create_rock_track(data.topic, duration_sec=data.duration_sec)
        logger.info(f"✅ Done: {result}")
        return {"status": "success", **result, "url": "/download"}

    except torch.cuda.OutOfMemoryError as e:
        free_vram()
        tb = traceback.format_exc()
        logger.error(f"💥 CUDA OOM:\n{tb}")
        raise HTTPException(status_code=507, detail=f"CUDA OOM: {e}")

    except Exception as e:
        free_vram()
        tb = traceback.format_exc()
        logger.error(f"💥 Error in generation:\n{tb}")
        # Возвращаем traceback в ответ, чтобы видеть в Swagger UI
        raise HTTPException(
            status_code=500,
            detail=f"{type(e).__name__}: {e}\n\nTraceback:\n{tb}"
        )

    finally:
        _gen_lock.release()


@app.get("/download")
def api_download():
    if not os.path.exists(OUTPUT_FILE):
        raise HTTPException(status_code=404, detail="No track generated yet")
    return FileResponse(OUTPUT_FILE, media_type="audio/wav", filename=OUTPUT_FILE)


@app.get("/health")
def api_health():
    info = {"status": "ok", "device": DEVICE}
    if DEVICE == "cuda":
        info["vram_used_gb"] = round(torch.cuda.memory_allocated() / 1e9, 2)
        info["vram_total_gb"] = round(
            torch.cuda.get_device_properties(0).total_memory / 1e9, 2
        )
        info["gpu_name"] = torch.cuda.get_device_name(0)
    return info


@app.get("/test")
def api_test():
    """Быстрый тест без генерации — проверяет, что сервер вообще жив."""
    return {"status": "alive", "message": "Server is running"}

# ==========================================
# 9. ЗАПУСК
# ==========================================
def start_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")


try:
    token = userdata.get("NGROK_TOKEN")
    ngrok.set_auth_token(token)
    ngrok.kill()
    public_url = ngrok.connect(8000).public_url
    print("★" * 60)
    print(f"🚀 API:    {public_url}/docs")
    print(f"📥 Health: {public_url}/health")
    print("★" * 60)
except Exception as e:
    print(f"⚠️ Ngrok error: {e}")

threading.Thread(target=start_server, daemon=True).start()

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
💻 Device: cuda | dtype: torch.float16
🚀 Загружаю модели (3-5 минут)...


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
/usr/local/lib/python3.12/dist-packages/transformers/models/encodec/modeling_encodec.py:120: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.register_buffer("padding_total", torch.tensor(kernel_size - stride, dtype=torch.int6

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

  💾 [after load] VRAM used=4.44 GB | peak=8.09 GB
✅ Студия готова!

★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
🚀 API:    https://outboard-clerk-twiddle.ngrok-free.dev/docs
📥 Health: https://outboard-clerk-twiddle.ngrok-free.dev/health
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★


In [ ]:
!nvidia-smi


Sun Apr 26 19:23:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   34C    P0             58W /  400W |    4780MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
result = create_rock_track("Большие города", duration_sec=12)
print(result)

  💾 [start] VRAM used=4.44 GB | peak=4.44 GB
📝 Пишу текст...


/usr/local/lib/python3.12/dist-packages/bark/generation.py:175: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with InferenceContext(), torch.inference_mode(), torch.no_grad(), autocast():


   → Улицы стучат шагами, / Многоногий город звонит. / Звуки твоего сердца гудят, / В огромных улицах волны.
  💾 [after lyrics] VRAM used=4.45 GB | peak=8.14 GB
🎤 Записываю вокал...


100%|██████████| 20/20 [00:14<00:00,  1.35it/s]


  💾 [after vocals] VRAM used=0.01 GB | peak=8.14 GB
🎸 Генерирую аранжировку (~12 сек)...
  💾 [after music] VRAM used=0.06 GB | peak=8.39 GB
🎚️ Свожу...
✅ Готово: 12.9 сек

{'lyrics': 'Улицы стучат шагами, / Многоногий город звонит. / Звуки твоего сердца гудят, / В огромных улицах волны.', 'duration_sec': 12.94}
